# Normalización del dataset de incendios de Galicia

## 1 - Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import unicodedata
import os
from difflib import get_close_matches

## 2 - Carga del dataset

In [ ]:
# Ruta del archivo de incendios
dataset_path = r'C:\00 - Proyecto Incendios Galicia 8.0\data\01 - Originales\02 - incendios\01 - Historico incendios galicia.xlsx'

# Cargar el archivo Excel (por defecto lee la primera hoja)
df = pd.read_excel(dataset_path)

# Normalizar nombres de columna a minúsculas para estandarizar el dataset
df.columns = [col.lower() for col in df.columns]
print('Nombres de columna normalizados a minúsculas:')
print(df.columns.tolist())

print(f"Filas cargadas: {len(df)}")

Nombres de columna normalizados a minúsculas:
['campania', 'numeroparte', 'estado', 'comunidad', 'provincia', 'municipio', 'comarcaisla', 'entidadmenor', 'numeromunicipiosafectados', 'hoja', 'cuadricula', 'huso', 'coordenadax', 'coordenaday', 'datum', 'numeropuntosinicioincendio', 'detectado', 'extinguido', 'causa', 'motivacion', 'superficiearbolada', 'superficienoarbolada', 'superficietotalforestal', 'superficieagricola', 'otrassuperficiesnoforestales', 'afectozonasinterfazurbanoforestal', 'tipointerfazafectado', 'afectoespacioprotegido', 'afectotierrasagrarias', 'afectozar', 'numeropartepss']
Filas cargadas: 113940


## 3 - Visualización y exploración inicial

In [3]:
# Ver las primeras filas del dataset
df.head()

# Ver información general del dataset
df.info()

# Ver estadísticas básicas de las columnas numéricas
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113940 entries, 0 to 113939
Data columns (total 31 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   campania                           113939 non-null  float64
 1   numeroparte                        113939 non-null  float64
 2   estado                             113939 non-null  object 
 3   comunidad                          113939 non-null  object 
 4   provincia                          113939 non-null  object 
 5   municipio                          113939 non-null  object 
 6   comarcaisla                        113939 non-null  object 
 7   entidadmenor                       113939 non-null  object 
 8   numeromunicipiosafectados          113939 non-null  float64
 9   hoja                               113939 non-null  float64
 10  cuadricula                         113939 non-null  object 
 11  huso                               1139

,campania,numeroparte,numeromunicipiosafectados,hoja
count,113939.000000,1.139390e+05,113939.000000,113939.000000
mean,2006.503857,2.006790e+09,1.001553,182.858705
std,5.558542,5.556355e+06,0.052054,57.162961
min,2000.000000,2.000150e+09,1.000000,0.000000
25%,2002.000000,2.002322e+09,1.000000,102.000000
50%,2005.000000,2.005321e+09,1.000000,202.000000
75%,2011.000000,2.011150e+09,1.000000,202.000000
max,2022.000000,2.022150e+09,8.000000,302.000000


## 4 - Cargar el dataset limpio de municipios de Galicia

In [ ]:
# Ruta del archivo de municipios limpios
municipios_path = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\01 - municipios\01 - Tabla de municipios.xlsx'

# Cargar el archivo Excel de municipios
df_municipios = pd.read_excel(municipios_path)

# Mostrar las primeras filas para comprobar que se ha cargado bien
df_municipios.head()

,municipio,comarca,provincia,altitud,superficie,poblacion,densidad
0,a arnoia,comarca del ribeiro,ourense,76,"20,69",1.000,"48,33"
1,a baña,barcala,a coruña,297,"98,19",3.450,"35,14"
2,a bola,comarca de tierra de celanova,ourense,510,"34,9",1.156,"33,12"
3,a capela,comarca del eume,a coruña,NaN,58,1.232,"21,24"
4,a cañiza,comarca de paradanta,pontevedra,570,"105,04",5.180,"49,31"


## 5 - Extraer y mostrar los nombres únicos de municipios de la tabla de referencia

In [5]:
# Extraer los nombres únicos de municipios de la tabla de referencia
nombres_municipios = df_municipios['municipio'].unique()

# Mostrar cuántos municipios únicos hay y los primeros 10 nombres
print(f"Municipios únicos en la referencia: {len(nombres_municipios)}")
print(nombres_municipios[:10])

Municipios únicos en la referencia: 315
['a arnoia' 'a baña' 'a bola' 'a capela' 'a cañiza' 'a coruña' 'a estrada'
 'a fonsagrada' 'a guarda' 'a gudiña']


## 6 - Extraer y mostrar los nombres únicos de municipios en el dataset de incendios

In [6]:
# Extraer los nombres únicos de municipios en el dataset de incendios
nombres_incendios = df['municipio'].unique()

# Mostrar cuántos municipios únicos hay y los primeros 10 nombres
print(f"Municipios únicos en el dataset de incendios: {len(nombres_incendios)}")
print(nombres_incendios[:10])

Municipios únicos en el dataset de incendios: 321
[nan 'FISTERRA' 'BOIRO' 'MONFERO' 'ORTIGUEIRA' 'ZAS' 'VAL DO DUBRA'
 'VIMIANZO' 'CORUÑA, A' 'RIANXO']


## 7 - Normalización y exportación de municipios
En este apartado se realiza todo el flujo de normalización y exportación de municipios, paso a paso para máxima trazabilidad y reproducibilidad.

### 7.1 Normalización automática y exportación a txt

In [ ]:
# 1. Normalización automática de municipios y exportación de cambios

ruta_txt = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\02 - incendios'
os.makedirs(ruta_txt, exist_ok=True)
def normalizar_nombre(nombre):
    if pd.isnull(nombre):
        return ''
    nombre = str(nombre).strip().lower()
    nombre = ''.join(c for c in unicodedata.normalize('NFD', nombre) if unicodedata.category(c) != 'Mn')
    nombre = nombre.replace('-', ' ').replace(',', '').replace('.', '')
    nombre = ' '.join(nombre.split())
    return nombre

# Diccionario de referencia normalizada
ref_norm = {normalizar_nombre(x): x for x in df_municipios['municipio'].unique()}
ref_norm_keys = set(ref_norm.keys())

def sugerencia_o_original(m):
    clave = normalizar_nombre(m)
    if clave in ref_norm:
        return ref_norm[clave]
    sugerencias = get_close_matches(clave, ref_norm_keys, n=1, cutoff=0.8)
    if sugerencias:
        return ref_norm[sugerencias[0]]
    return m

# Crear columna adicional 'Municipio_normalizado' (no sobrescribe la original)
df['municipio_normalizado'] = df['municipio'].apply(sugerencia_o_original)

# Municipios originales y normalizados
cambios = sorted(set((o, n) for o, n in zip(df['municipio'], df['municipio_normalizado']) if isinstance(o, str) and o.strip() and o != n))

# Municipios pendientes (no normalizados)
originales = set(x for x in df['municipio'].unique() if isinstance(x, str) and x.strip())
normalizados = set(df['municipio_normalizado'].unique())
pendientes = sorted([x for x in originales if normalizar_nombre(x) not in ref_norm and not get_close_matches(normalizar_nombre(x), ref_norm_keys, n=1, cutoff=0.8)])

print(f"Municipios normalizados automáticamente: {len(cambios)}")
print(f"Municipios pendientes de normalizar: {len(pendientes)}")

# Exportar cambios a txt (2 columnas, tabuladas)
archivo_cambios = os.path.join(ruta_txt, 'municipios_normalizados_automatica.txt')
with open(archivo_cambios, 'w', encoding='utf-8') as f:
    for orig, norm in cambios:
        f.write(f'{orig}\t{norm}\n')
print(f'Archivo {archivo_cambios} generado (2 columnas, tabuladas)')

# Exportar municipios pendientes de normalizar (para revisión manual)
archivo_pendientes = os.path.join(ruta_txt, 'municipios_no_normalizados_final.txt')
with open(archivo_pendientes, 'w', encoding='utf-8') as f:
    for m in pendientes:
        f.write(f'{m}\n')
print(f'Municipios pendientes de normalizar manualmente: {len(pendientes)}')
print(f'Listado exportado a {archivo_pendientes}')

Municipios normalizados automáticamente: 283
Municipios pendientes de normalizar: 37
Archivo C:\00 - Proyecto Incendios Galicia 7.0\data\02 - Municipio normalizado\02 - incendios\municipios_normalizados_automatica.txt generado (2 columnas, tabuladas)
Municipios pendientes de normalizar manualmente: 37
Listado exportado a C:\00 - Proyecto Incendios Galicia 7.0\data\02 - Municipio normalizado\02 - incendios\municipios_no_normalizados_final.txt


### 7.2 Normalización manual

In [ ]:
# Aplicar la normalización manual desde el archivo corregido y mostrar los que aún quedan sin normalizar

ruta_txt = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\02 - incendios'
archivo_manual = os.path.join(ruta_txt, 'municipios_normalizados_a_mano.txt')
manual_map = {} # Diccionario vacío para almacenar las normalizaciones manuales
with open(archivo_manual, 'r', encoding='utf-8') as f:
    for line in f:  # Recorre cada línea del archivo
        parts = line.rstrip().split('\t')   # Separa la línea en dos partes por tabulación
        if len(parts) == 2 and parts[1].strip():    # Separa la línea en dos partes por tabulación
            manual_map[parts[0].strip()] = parts[1].strip() # Añade la correspondencia al diccionario

def aplicar_manual(row):    # Función para aplicar la normalización manual a cada fila
    orig = row['municipio_normalizado'] # Ahora toma el nombre ya normalizado automáticamente
    if orig in manual_map:  # Si el municipio está en el diccionario manual
        return manual_map[orig] # Devuelve el nombre corregido
    return orig # Si no, devuelve el nombre tal cual

# Crear columna adicional para la normalización final
df['municipio_normalizado_final'] = df.apply(aplicar_manual, axis=1)    # Aplica la función a todo el DataFrame, sobrescribiendo la columna

# Municipios que siguen sin normalizar tras la corrección manual (solo los que no están en la referencia oficial)
referencia_oficial = set(df_municipios['municipio'].unique())
restantes = []
for x in df['municipio_normalizado_final'].unique():
    if not isinstance(x, str) or not x.strip():
        continue
    if x not in referencia_oficial:
        restantes.append(x)
print(f'Municipios que siguen sin normalizar tras la corrección manual: {len(restantes)}')
for m in restantes:
    print(m)

Municipios que siguen sin normalizar tras la corrección manual: 1
ELIMINAR


### 7.3 Filtrar municipios 'ELIMINAR', dejar la columna 'Municipio' normalizada y contar municipios únicos

In [9]:
# Eliminar filas cuyo municipio final es 'ELIMINAR'
df_filtrado = df[df['municipio_normalizado_final'] != 'ELIMINAR'].copy()

# Sustituir la columna 'Municipio' por la columna normalizada final
df_filtrado['municipio'] = df_filtrado['municipio_normalizado_final']

# Eliminar columnas auxiliares si existen
for col in ['municipio_normalizado', 'municipio_normalizado_final']:
    if col in df_filtrado.columns:
        df_filtrado = df_filtrado.drop(columns=[col])

# Contar municipios únicos en el dataset final
municipios_unicos = df_filtrado['municipio'].nunique()
print(f'Municipios únicos en el dataset de incendios tras filtrar y normalizar: {municipios_unicos}')

# Mostrar los primeros municipios únicos para comprobación
print('Ejemplo de municipios únicos:')
print(df_filtrado['municipio'].unique()[:10])

Municipios únicos en el dataset de incendios tras filtrar y normalizar: 314
Ejemplo de municipios únicos:
[nan 'fisterra' 'boiro' 'monfero' 'ortigueira' 'zas' 'val do dubra'
 'vimianzo' 'a coruña' 'rianxo']


### 7.4 Diagnóstico: filas con municipio NaN y municipios faltantes respecto a la referencia oficial

In [10]:
# Mostrar las filas con municipio NaN
filas_nan = df_filtrado[df_filtrado['municipio'].isna()]
print(f'Filas con Municipio NaN: {len(filas_nan)}')
if not filas_nan.empty:
    display(filas_nan)

# Municipios únicos en el dataset filtrado
municipios_filtrados = set(df_filtrado['municipio'].dropna().unique())

# Municipios únicos en la referencia oficial
municipios_referencia = set(df_municipios['municipio'].unique())

# Municipios de la referencia que faltan en el dataset filtrado
faltantes = municipios_referencia - municipios_filtrados
print(f'Municipios de la referencia que faltan en el dataset filtrado: {len(faltantes)}')
print(faltantes)

Filas con Municipio NaN: 1


,campania,numeroparte,estado,comunidad,provincia,municipio,comarcaisla,entidadmenor,numeromunicipiosafectados,hoja,...,superficienoarbolada,superficietotalforestal,superficieagricola,otrassuperficiesnoforestales,afectozonasinterfazurbanoforestal,tipointerfazafectado,afectoespacioprotegido,afectotierrasagrarias,afectozar,numeropartepss
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Municipios de la referencia que faltan en el dataset filtrado: 1
{'cerdedo-cotobade'}


### 7.5 Eliminar filas con municipio NaN

In [11]:
# Eliminar filas con municipio NaN en la columna 'municipio'
df_filtrado = df_filtrado[~df_filtrado['municipio'].isna()].copy()
print(f'Filas restantes tras eliminar NaN en municipio: {len(df_filtrado)}')

Filas restantes tras eliminar NaN en municipio: 113828


### 7.6 Añadir registros combinados para el municipio unificado 'Cerdedo-Cotobade' (315 municipios)

In [ ]:
# Obtener el nombre exacto de la referencia oficial para el municipio unificado
nombre_unificado = [m for m in df_municipios['municipio'].unique() if 'cerdedo' in m.lower() and 'cotobade' in m.lower()]
if nombre_unificado:
    nombre_unificado = nombre_unificado[0]
else:
    nombre_unificado = 'Cerdedo-Cotobade'  # Fallback por si no se encuentra

# Seleccionar registros de Cerdedo y Cotobade (ignorando mayúsculas/minúsculas y tildes)
def normalizar_simple(nombre):
    
    if pd.isnull(nombre): return ''
    nombre = str(nombre).strip().lower()
    nombre = ''.join(c for c in unicodedata.normalize('NFD', nombre) if unicodedata.category(c) != 'Mn')
    return nombre

df_cerdedo = df_filtrado[df_filtrado['municipio'].apply(lambda x: normalizar_simple(x) == normalizar_simple('Cerdedo'))]
df_cotobade = df_filtrado[df_filtrado['municipio'].apply(lambda x: normalizar_simple(x) == normalizar_simple('Cotobade'))]

# Duplicar y renombrar municipio al nombre exacto de la referencia oficial
df_cerdedo_cotobade = pd.concat([df_cerdedo, df_cotobade]).copy()
df_cerdedo_cotobade['municipio'] = nombre_unificado

# Concatenar al DataFrame filtrado
df_final = pd.concat([df_filtrado, df_cerdedo_cotobade], ignore_index=True)

# Comprobar el número de municipios únicos
municipios_unicos_final = df_final['municipio'].nunique()
print(f'Municipios únicos tras añadir {nombre_unificado}: {municipios_unicos_final}')
print('Ejemplo de municipios únicos:')
print(df_final['municipio'].unique()[:10])

Municipios únicos tras añadir cerdedo-cotobade: 315
Ejemplo de municipios únicos:
['fisterra' 'boiro' 'monfero' 'ortigueira' 'zas' 'val do dubra' 'vimianzo'
 'a coruña' 'rianxo' 'santa comba']


### 7.7 Normalizamos nuevamente los nombres de columna

In [13]:
# Normalizar nombres de columna a minúsculas para estandarizar el dataset
df.columns = [col.lower() for col in df.columns]
print('Nombres de columna normalizados a minúsculas:')
print(df.columns.tolist())

Nombres de columna normalizados a minúsculas:
['campania', 'numeroparte', 'estado', 'comunidad', 'provincia', 'municipio', 'comarcaisla', 'entidadmenor', 'numeromunicipiosafectados', 'hoja', 'cuadricula', 'huso', 'coordenadax', 'coordenaday', 'datum', 'numeropuntosinicioincendio', 'detectado', 'extinguido', 'causa', 'motivacion', 'superficiearbolada', 'superficienoarbolada', 'superficietotalforestal', 'superficieagricola', 'otrassuperficiesnoforestales', 'afectozonasinterfazurbanoforestal', 'tipointerfazafectado', 'afectoespacioprotegido', 'afectotierrasagrarias', 'afectozar', 'numeropartepss', 'municipio_normalizado', 'municipio_normalizado_final']


### 7.8 Guardado del archivo final con municipios normalizados

In [ ]:
# Guardar el DataFrame final con municipios normalizados

output_csv = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\02 - incendios\01 - Historico incendios galicia municipio normalizado.csv'

# Usar df_final, que ya no contiene municipios 'ELIMINAR' y tiene los registros combinados de Cerdedo-Cotobade
df_export = df_final.copy()

df_export.to_csv(output_csv, index=False, encoding='utf-8')
print(f'Archivo guardado en: {output_csv}')

Archivo guardado en: C:\00 - Proyecto Incendios Galicia 7.0\data\02 - Municipio normalizado\02 - incendios\01 - Historico incendios galicia municipio normalizado.csv


### 7.9 Hacemos una última comprobación

In [ ]:
# Comprobación final de municipios únicos y coincidencia con la referencia oficial

# Cargar el archivo final generado
archivo_final = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\02 - incendios\01 - Historico incendios galicia municipio normalizado.csv'
df_final = pd.read_csv(archivo_final)

# Cargar la referencia oficial de municipios
archivo_municipios = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\01 - municipios\01 - Tabla de municipios.csv'
df_municipios = pd.read_csv(archivo_municipios)

# Extraer municipios únicos de ambos datasets
municipios_final = set(df_final['municipio'].dropna().unique())
municipios_referencia = set(df_municipios['municipio'].dropna().unique())

print(f"Municipios únicos en el archivo final: {len(municipios_final)}")
print(f"Municipios únicos en la referencia oficial: {len(municipios_referencia)}")

# Comprobar si coinciden al 100%
if municipios_final == municipios_referencia:
    print("Los municipios coinciden al 100% con la referencia oficial.")
else:
    print("Los municipios NO coinciden al 100%.")
    print(f"Municipios en la referencia que faltan en el archivo final: {municipios_referencia - municipios_final}")
    print(f"Municipios en el archivo final que no están en la referencia: {municipios_final - municipios_referencia}")

Municipios únicos en el archivo final: 315
Municipios únicos en la referencia oficial: 315
Los municipios coinciden al 100% con la referencia oficial.


C:\Users\Jacinto\AppData\Local\Temp\ipykernel_25056\3523789475.py:5: DtypeWarning: Columns (11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df_final = pd.read_csv(archivo_final)
